# Goal Creation Patterns

Companion notebook for the [Goal Creation lesson](https://ml-viz-ruby.vercel.app/courses/agent-design-patterns/02-goal-creation-patterns).

**The idea in one sentence.** Before an agent can plan, it needs a well-formed **goal** —
and the two patterns differ in initiative: a **passive** goal creator takes the user's
request at face value, while a **proactive** one detects ambiguity and *asks clarifying
questions* before committing.

The tradeoff: passive is fast and cheap but acts on under-specified requests (and gets
them wrong); proactive costs an extra turn but avoids expensive misfires on ambiguous
goals. The right choice depends on the cost of being wrong.

We build both patterns, add **what to notice** on when each wins, then cover the
gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import re
import json
from dataclasses import dataclass, field
from typing import Optional
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor':  '#0f1117',
    'axes.facecolor':    '#1a1d27',
    'axes.edgecolor':    '#2d3148',
    'axes.labelcolor':   '#e2e8f0',
    'xtick.color':       '#94a3b8',
    'ytick.color':       '#94a3b8',
    'text.color':        '#e2e8f0',
    'grid.color':        '#2d3148',
    'lines.linewidth':   1.5,
    'font.size':         11,
})
BRAND  = '#6366f1'
TEAL   = '#2dd4bf'
ROSE   = '#fb7185'
YELLOW = '#fbbf24'
MUTED  = '#475569'

## Passive Goal Creator

The Passive Goal Creator converts an **explicit user message** into a structured goal object. The core challenge is NLP: extracting intent, entities, and constraints from free-form text.

Production implementations use an FM for this parsing step. Here we use regex and keyword matching to keep the example self-contained and transparent.

In [ ]:
@dataclass
class GoalObject:
    intent: str
    entities: dict = field(default_factory=dict)
    constraints: dict = field(default_factory=dict)
    clarification_needed: Optional[str] = None

    def __repr__(self):
        parts = [f"intent={self.intent!r}"]
        if self.entities:
            parts.append(f"entities={self.entities}")
        if self.constraints:
            parts.append(f"constraints={self.constraints}")
        if self.clarification_needed:
            parts.append(f"clarification_needed={self.clarification_needed!r}")
        return f"GoalObject({', '.join(parts)})"


class PassiveGoalCreator:
    """
    Converts explicit user messages into structured GoalObjects.
    Uses keyword + regex matching as a proxy for FM-based intent extraction.
    """

    INTENT_PATTERNS = [
        (r'\b(book|reserve|schedule)\b.*(flight|plane|ticket)', 'book_flight'),
        (r'\b(book|reserve|find)\b.*(hotel|accommodation|room)',  'book_hotel'),
        (r'\b(remind|reminder)\b',                               'set_reminder'),
        (r'\b(send|email|mail)\b',                               'send_email'),
        (r'\b(search|find|look up|what is)\b',                   'information_lookup'),
        (r'\b(cancel|delete|remove)\b',                          'cancel_item'),
    ]

    def parse(self, message: str) -> GoalObject:
        message_lower = message.lower()

        # Intent detection
        intent = 'unknown'
        for pattern, intent_name in self.INTENT_PATTERNS:
            if re.search(pattern, message_lower):
                intent = intent_name
                break

        # Entity extraction
        entities = {}
        dest_match = re.search(r'\bto\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)\b', message)
        if dest_match:
            entities['destination'] = dest_match.group(1)

        date_match = re.search(r'\b(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday|tomorrow|today|next week|Friday)\b',
                               message, re.IGNORECASE)
        if date_match:
            entities['date'] = date_match.group(1)

        # Constraint extraction
        constraints = {}
        price_match = re.search(r'\$([\d,]+)', message)
        if price_match:
            constraints['max_price_usd'] = int(price_match.group(1).replace(',', ''))

        # Clarification needed?
        clarification = None
        if intent == 'set_reminder' and 'meeting' in message_lower and 'date' not in entities:
            clarification = "Which meeting should I set a reminder for? Please specify date and time."

        return GoalObject(intent=intent, entities=entities, constraints=constraints,
                         clarification_needed=clarification)


pgc = PassiveGoalCreator()

test_messages = [
    "Book me a flight to Paris on Friday under $500",
    "Remind me about my meeting",
    "Find the best Python tutorials online",
]

for msg in test_messages:
    goal = pgc.parse(msg)
    print(f"Input : {msg!r}")
    print(f"Output: {goal}")
    print()

## Proactive Goal Creator

The Proactive Goal Creator monitors a **context stream** — a sequence of events that represent user behaviour — and infers goals without the user explicitly asking.

Key design decisions:
- **What signals to monitor** (calendar, searches, file access, communication metadata)
- **Confidence threshold** — below this, the inferred goal is discarded or queued
- **Suggest vs auto-execute** — whether to propose the goal to the user first

In [ ]:
@dataclass
class ContextEvent:
    signal_type: str   # 'search', 'calendar', 'file_access', 'communication'
    content: str
    timestamp: str


class ProactiveGoalCreator:
    """
    Monitors a context stream and infers goals above a confidence threshold.
    In production, the inference engine would be an FM call.
    """

    def __init__(self, confidence_threshold: float = 0.7, mode: str = 'suggest'):
        """
        confidence_threshold: minimum confidence to surface a goal
        mode: 'suggest' (propose to user) or 'auto-execute' (act immediately)
        """
        assert mode in ('suggest', 'auto-execute'), "mode must be 'suggest' or 'auto-execute'"
        self.threshold = confidence_threshold
        self.mode = mode

    def _infer_goals(self, events: list) -> list:
        """
        Infer candidate goals from a list of context events.
        Returns list of (goal_description, confidence) tuples.
        """
        candidates = []
        contents = [e.content.lower() for e in events]
        types    = [e.signal_type for e in events]

        # Pattern: repeated weather searches for a city → travel intent
        weather_searches = [c for c, t in zip(contents, types)
                            if t == 'search' and 'weather' in c]
        if len(weather_searches) >= 2:
            cities = [re.search(r'weather in (\w+)', c) for c in weather_searches]
            cities = [m.group(1).capitalize() for m in cities if m]
            if cities:
                city = max(set(cities), key=cities.count)  # most common city
                confidence = min(0.5 + 0.15 * len(weather_searches), 0.95)
                candidates.append((f"Search flights to {city}", confidence))

        # Pattern: calendar event tomorrow + no preparation found
        tomorrow_events = [c for c, t in zip(contents, types)
                           if t == 'calendar' and 'tomorrow' in c]
        if tomorrow_events:
            for event_desc in tomorrow_events:
                candidates.append((f"Prepare for: {event_desc}", 0.80))

        return candidates

    def trigger_goal(self, events: list) -> list:
        """
        Process context events and return triggered goals (above threshold).
        """
        candidates = self._infer_goals(events)
        triggered  = [(goal, conf) for goal, conf in candidates if conf >= self.threshold]

        results = []
        for goal, conf in triggered:
            action = 'SUGGEST' if self.mode == 'suggest' else 'AUTO-EXECUTE'
            results.append({'goal': goal, 'confidence': conf, 'action': action})
        return results


proactive = ProactiveGoalCreator(confidence_threshold=0.7, mode='suggest')

# Scenario B: user has been checking Paris weather for 3 days
paris_weather_stream = [
    ContextEvent('search',   'weather in paris',    '2026-06-16 09:00'),
    ContextEvent('search',   'weather in paris',    '2026-06-17 08:45'),
    ContextEvent('search',   'weather in paris',    '2026-06-18 09:10'),
    ContextEvent('calendar', 'team standup tomorrow', '2026-06-18 10:00'),
]

triggered = proactive.trigger_goal(paris_weather_stream)
print("Triggered goals from context stream:")
for t in triggered:
    print(f"  [{t['action']}] {t['goal']!r}  (confidence={t['confidence']:.2f})")

In [ ]:
# Compare both patterns across three scenarios

scenarios = [
    {
        'name': 'A — Explicit flight request',
        'message': 'Book me a flight to Paris on Friday under $500',
        'context': [],  # no context events
    },
    {
        'name': 'B — Implicit travel intent (3-day weather pattern)',
        'message': None,  # no explicit message
        'context': [
            ContextEvent('search', 'weather in paris', '2026-06-16 09:00'),
            ContextEvent('search', 'weather in paris', '2026-06-17 08:45'),
            ContextEvent('search', 'weather in paris', '2026-06-18 09:10'),
        ],
    },
    {
        'name': 'C — Ambiguous reminder request',
        'message': 'remind me about my meeting',
        'context': [
            ContextEvent('calendar', 'board meeting tomorrow at 10am', '2026-06-18 09:00'),
        ],
    },
]

pgc_inst  = PassiveGoalCreator()
pro_inst  = ProactiveGoalCreator(confidence_threshold=0.7, mode='suggest')

print(f"{'Scenario':<42} {'Passive result':<55} {'Proactive result'}")
print("-" * 140)

for s in scenarios:
    passive_result = pgc_inst.parse(s['message']) if s['message'] else '(no message)'
    proactive_result = pro_inst.trigger_goal(s['context']) if s['context'] else []

    passive_str   = str(passive_result)[:52] + '...' if len(str(passive_result)) > 52 else str(passive_result)
    proactive_str = str(proactive_result)[:60] + '...' if len(str(proactive_result)) > 60 else str(proactive_result)

    print(f"{s['name']:<42} {passive_str:<55} {proactive_str}")

**What to notice — initiative vs cost.** On a *clear* request both patterns produce
the same goal, and the passive one is cheaper (no extra turn). On an *ambiguous*
request the passive creator commits to a guess while the proactive one asks a
clarifying question first. The confidence threshold is the dial: lower it and the agent
interrupts more often (safer, chattier); raise it and it assumes more (faster,
riskier).

In [ ]:
# Visualise confidence threshold effect on proactive goal triggering

n_weather_events = list(range(1, 8))
confidence_by_n  = [min(0.5 + 0.15 * n, 0.95) for n in n_weather_events]

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(n_weather_events, confidence_by_n, color=BRAND, marker='o', ms=7, label='Inferred confidence')
for threshold, color, label in [(0.7, TEAL, 'threshold=0.70'), (0.85, YELLOW, 'threshold=0.85')]:
    ax.axhline(threshold, color=color, ls='--', lw=1.5, label=label)
    # shade triggered region
    cross = next((n for n, c in zip(n_weather_events, confidence_by_n) if c >= threshold), None)
    if cross:
        ax.axvspan(cross - 0.5, max(n_weather_events) + 0.5, alpha=0.07, color=color)

ax.set_xlabel('Number of "weather in Paris" searches observed')
ax.set_ylabel('Confidence score')
ax.set_title('Proactive Goal Creator: confidence vs observed signal count')
ax.set_xticks(n_weather_events)
ax.legend()
plt.tight_layout()
plt.show()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **passive on ambiguous input** | commits to a wrong guess; expensive when actions are irreversible |
| **proactive over-clarifying** | annoys users with questions on clear requests — tune the threshold |
| **mis-calibrated confidence** | a badly-calibrated ambiguity score triggers at the wrong times |
| **no goal validation** | a malformed goal propagates errors into planning and execution |

Demo: proactive clarification wins when a wrong action costs more than an extra turn.

In [ ]:
# The core tradeoff quantified: a proactive clarification costs one extra turn but
# avoids acting on a wrong guess. Whether it pays off depends on P(ambiguous) and the
# cost of a wrong action vs the cost of an extra turn.
def expected_cost(p_ambiguous, wrong_cost, clarify_cost, proactive):
    if proactive:
        # always pay clarify_cost on ambiguous inputs, never act wrongly
        return p_ambiguous * clarify_cost
    # passive: act on the guess; wrong on ambiguous inputs
    return p_ambiguous * wrong_cost
for p_amb in [0.1, 0.3, 0.6]:
    passive = expected_cost(p_amb, wrong_cost=10.0, clarify_cost=1.0, proactive=False)
    proactive = expected_cost(p_amb, wrong_cost=10.0, clarify_cost=1.0, proactive=True)
    better = 'proactive' if proactive < passive else 'passive'
    print(f'P(ambiguous)={p_amb}: passive cost {passive:.1f}, proactive cost {proactive:.1f} -> {better} wins')
print('\nWhen wrong actions are expensive (>clarify cost), proactive clarification pays off.')

## ✏️ Your turn

**Exercise — Implement a `GoalDecomposer`.**

Both patterns produce a top-level `GoalObject`. Before planning can begin, this must be decomposed into 3–5 instrumental subgoals.

Your task:
1. Implement a `GoalDecomposer` class with a `decompose(goal: GoalObject) -> list[str]` method
2. It should return 3–5 ordered subgoal strings for a `book_flight` or `book_hotel` intent
3. Test on: `GoalObject(intent='book_flight', entities={'destination': 'Tokyo'}, constraints={'max_price_usd': 800})`
4. Assert the output has 3–5 items and each is a non-empty string

In [ ]:
class GoalDecomposer:
    """
    Decomposes a top-level GoalObject into a list of instrumental subgoals.
    In production this would be an FM call; here use a rule-based approach.
    """

    # TODO(you): implement decompose
    def decompose(self, goal: GoalObject) -> list:
        """
        Returns a list of 3-5 ordered subgoal strings.
        """
        pass  # replace with your implementation


tokyo_goal = GoalObject(
    intent='book_flight',
    entities={'destination': 'Tokyo'},
    constraints={'max_price_usd': 800},
)

decomposer = GoalDecomposer()
subgoals = decomposer.decompose(tokyo_goal)

print("Top-level goal:", tokyo_goal)
print("\nInstrumental subgoals:")
if subgoals:
    for i, sg in enumerate(subgoals, 1):
        print(f"  {i}. {sg}")
else:
    print("  (implement decompose() to see results)")

In [ ]:
# Assertion cell — runs silently when correct
assert subgoals is not None, "decompose() must return a list, not None"
assert isinstance(subgoals, list), "decompose() must return a list"
assert 3 <= len(subgoals) <= 5, \
    f"Expected 3-5 subgoals, got {len(subgoals)}: {subgoals}"
assert all(isinstance(s, str) and len(s.strip()) > 0 for s in subgoals), \
    "All subgoals must be non-empty strings"

# At minimum, the decomposition should mention the destination
all_text = ' '.join(subgoals).lower()
assert 'tokyo' in all_text or 'flight' in all_text or 'search' in all_text, \
    "Subgoals should reference the destination or flight-related steps"

print("✓ Exercise passed — GoalDecomposer returns valid subgoals")

<details>
<summary>Solution</summary>

```python
class GoalDecomposer:
    DECOMPOSITIONS = {
        'book_flight': [
            "Search for available flights to {destination} within budget ${max_price_usd}",
            "Compare top flight options by price, duration, and number of stops",
            "Select and book the best matching flight",
            "Retrieve booking confirmation and itinerary",
        ],
        'book_hotel': [
            "Search for hotels near {destination}",
            "Filter by price (under ${max_price_usd}) and rating",
            "Select and reserve the top result",
            "Retrieve booking confirmation",
        ],
    }

    def decompose(self, goal: GoalObject) -> list:
        template_list = self.DECOMPOSITIONS.get(goal.intent, ["Execute top-level goal: " + goal.intent])
        context = {**goal.entities, **goal.constraints}
        result = []
        for tmpl in template_list:
            try:
                result.append(tmpl.format(**context))
            except KeyError:
                result.append(tmpl)
        return result
```
</details>

## Key takeaways

- **Goal creation precedes planning:** the agent must turn a request into a well-formed
  goal before it can act.
- **Passive = fast, takes requests at face value; proactive = asks first** when the
  request is ambiguous.
- **The confidence threshold is the dial** between interrupting (safe/chatty) and
  assuming (fast/risky).
- **Proactive clarification pays off when wrong actions are costly** relative to an
  extra turn (demo).